# PTCG AI - 重み付き模倣学習
リーダーボードスコアを重みとして使い、高スコアプレイヤーの行動を優先的に学習します。

In [ ]:
# ============================================================
# Step 1: パス自動検出
# ============================================================
import os, glob

INPUT_BASE = '/kaggle/input'
WORKING    = '/kaggle/working'

# デバッグ: /kaggle/input/ の構造を表示
print('=== /kaggle/input/ の構造 ===')
for root, dirs, files in os.walk(INPUT_BASE):
    depth = root.replace(INPUT_BASE, '').count(os.sep)
    if depth > 2:
        continue  # 深すぎる階層は省略
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or root}/  [{len(files)} files]')
    if depth == 2 and files:
        print(f'{indent}  first files: {files[:5]}')
print()

# /kaggle/input/ 以下を再帰的に検索して EN_Card_Data.csv を探す
CSV_PATH = None
for root, dirs, files in os.walk(INPUT_BASE):
    if 'EN_Card_Data.csv' in files:
        CSV_PATH = os.path.join(root, 'EN_Card_Data.csv')
        break

# /kaggle/input/ 以下を再帰的に検索して JSON ファイルを含むディレクトリを探す
# エピソードデータセット名（日付付き）を優先して探す
JSON_DIRECTORY = None
json_candidates = []
for root, dirs, files in os.walk(INPUT_BASE):
    json_files = [f for f in files if f.endswith('.json')]
    if json_files:
        json_candidates.append((root, len(json_files)))

if json_candidates:
    # JSON数が最も多いディレクトリを選択
    json_candidates.sort(key=lambda x: x[1], reverse=True)
    JSON_DIRECTORY = json_candidates[0][0]
    print(f'JSONディレクトリ候補: {json_candidates[:3]}')

# リーダーボードCSVを自動検出
LB_PATH = None
for root, dirs, files in os.walk(INPUT_BASE):
    for f in files:
        if 'leaderboard' in f.lower() and f.endswith('.csv'):
            LB_PATH = os.path.join(root, f)
            break

assert CSV_PATH,       'カードデータが見つかりません。コンペデータセットを追加してください。'
assert JSON_DIRECTORY, 'エピソードデータが見つかりません。エピソードデータセットを追加してください。'

archive_count = len(glob.glob(f'{JSON_DIRECTORY}/*.json'))
print(f'エピソードデータ: {JSON_DIRECTORY}')
print(f'JSONファイル数  : {archive_count}')
print(f'カードデータ    : {CSV_PATH}')
print(f'リーダーボード  : {LB_PATH if LB_PATH else "未検出（全勝者を均等学習）"}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# Step 2: 設定パラメータ
# ============================================================
# JSON_DIRECTORY / CSV_PATH は Step 1 で検出済み
LEADERBOARD_CSV = LB_PATH

MODEL_PATH = f'{WORKING}/ptcg_baseline_model.pth'
NORM_PATH  = f'{WORKING}/ptcg_normalization.npz'
PLOT_PATH  = f'{WORKING}/training_curve.png'

MAX_ACTIONS   = 256
MAX_FILES     = 1500  # None=全ファイル / 整数=メモリ節約（1500≒6GB peak）
BATCH_SIZE    = 512   # GPU使用時は大きく設定
EPOCHS        = 15
LEARNING_RATE = 1e-3
DROPOUT_RATE  = 0.3   # オーバーフィット対策
WEIGHT_DECAY  = 1e-4  # L2正則化

print('設定完了')

In [ ]:
# ============================================================
# Step 3: カードデータ読み込み & 特徴量関数
# ============================================================
import re
import numpy as np
import pandas as pd

df_cards = pd.read_csv(CSV_PATH)
df_cards['HP']       = df_cards['HP'].fillna(0).astype(float)
df_cards['Retreat']  = df_cards['Retreat'].fillna(0).astype(float)
df_cards['Type']     = df_cards['Type'].fillna('None')
df_cards['Weakness'] = df_cards['Weakness'].fillna('None')
df_cards['Damage']   = df_cards['Damage'].fillna('0')
df_cards['Cost']     = df_cards['Cost'].fillna('')
df_cards['Rule']     = df_cards['Rule'].fillna('')

df_cards_unique = df_cards.drop_duplicates(subset=['Card ID'], keep='first')
card_dict  = df_cards_unique.set_index('Card ID').to_dict(orient='index')
MAX_CARD_ID = int(df_cards['Card ID'].max())

TYPE_VOCAB     = ['{G}','{R}','{W}','{L}','{P}','{F}','{D}','{M}','{C}','竜',
                  '{A}','{A}{A}','{Team Rocket}{Team Rocket}','{C}{C}{C}']
WEAKNESS_VOCAB = ['{G}','{R}','{W}','{L}','{P}','{F}','{D}','{M}','{C}','竜']

def one_hot_encode(val, vocab):
    v = [0] * len(vocab)
    if val in vocab:
        v[vocab.index(val)] = 1
    return v

def parse_damage(s):  return int(re.findall(r'\d+', str(s))[0]) if re.findall(r'\d+', str(s)) else 0
def parse_cost(s):    return str(s).count('{') + str(s).count('●')

def encode_card_list(card_list, max_id):
    v = [0] * (max_id + 1)
    for c in (card_list or []):
        cid = c.get('id', 0)
        if 0 <= cid <= max_id:
            v[cid] += 1
    return v

def count_energy_types(energy_cards):
    counts = [0] * len(TYPE_VOCAB)
    for ec in energy_cards:
        etype = card_dict.get(ec.get('id', 0), {}).get('Type', 'None')
        if etype in TYPE_VOCAB:
            counts[TYPE_VOCAB.index(etype)] += 1
    return counts

def poke_feat(poke):
    empty = [0] * (7 + 2 + len(TYPE_VOCAB) + len(WEAKNESS_VOCAB) + len(TYPE_VOCAB))
    if not poke: return empty
    ci  = card_dict.get(poke.get('id', 0), {})
    return (
        [poke.get('id',0), poke.get('hp',0), ci.get('HP',0),
         len(poke.get('energies',[])), ci.get('Retreat',0),
         1 if 'ex' in str(ci.get('Rule','')).lower() else 0,
         (poke.get('tools') or [{}])[0].get('id', 0)]
        + [parse_damage(ci.get('Damage','0')), parse_cost(ci.get('Cost',''))]
        + one_hot_encode(ci.get('Type','None'), TYPE_VOCAB)
        + one_hot_encode(ci.get('Weakness','None'), WEAKNESS_VOCAB)
        + count_energy_types(poke.get('energyCards',[]))
    )

def player_feat(p):
    feats = []
    feats.extend(poke_feat((p.get('active') or [None])[0]))
    bench = p.get('bench', [])
    for i in range(5):
        feats.extend(poke_feat(bench[i] if i < len(bench) else None))
    feats.extend([int(p.get('poisoned',False)), int(p.get('burned',False)),
                  int(p.get('asleep',False)), int(p.get('paralyzed',False)),
                  int(p.get('confused',False))])
    prize = sum(1 for pr in p.get('prize',[]) if pr is not None)
    feats.extend([p.get('handCount',0), p.get('deckCount',0), prize])
    feats.extend(encode_card_list(p.get('hand',[]),    MAX_CARD_ID))
    feats.extend(encode_card_list(p.get('discard',[]), MAX_CARD_ID))
    return feats

def extract_state_vector(step_data):
    obs   = step_data.get('observation', {})
    cur   = obs.get('current')
    if not cur or len(cur.get('players', [])) < 2:
        return None
    stadium_id = (cur.get('stadium') or [{}])[0].get('id', 0)
    global_f = [stadium_id,
                int(cur.get('supporterPlayed', False)),
                int(cur.get('energyAttached',  False)),
                int(cur.get('retreated',        False)),
                int(cur.get('firstPlayer',     -1)),
                int(cur.get('turn',             0))]
    my = cur.get('yourIndex', 0)
    return np.array(global_f + player_feat(cur['players'][my])
                             + player_feat(cur['players'][1 - my]),
                    dtype=np.float32)

INPUT_DIM = 6 + 2 * (47 + 5*47 + 5 + 3 + (MAX_CARD_ID+1)*2)
print(f'MAX_CARD_ID: {MAX_CARD_ID}')
print(f'入力次元数 : {INPUT_DIM}')

In [ ]:
# ============================================================
# Step 4: モデル定義
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

class PTCGNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=1024, max_actions=256, dropout=0.3):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim)
        self.ln1   = nn.LayerNorm(hidden_dim)   # CPU では CUDA エラー不要
        self.drop1 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, hidden_dim // 2)
        self.ln2   = nn.LayerNorm(hidden_dim // 2)
        self.drop2 = nn.Dropout(dropout)
        self.fc3   = nn.Linear(hidden_dim // 2, max_actions)

    def forward(self, x):
        x = self.drop1(F.relu(self.ln1(self.fc1(x))))
        x = self.drop2(F.relu(self.ln2(self.fc2(x))))
        return self.fc3(x)

print('モデル定義完了')

In [ ]:
# ============================================================
# Step 5: Dataset 作成
# ============================================================
import json, glob
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

# リーダーボード読み込み（任意）
def load_score_map(csv_path):
    if csv_path is None:
        return None
    lb = pd.read_csv(csv_path)
    lb['TeamName'] = lb['TeamName'].str.strip()
    mn, mx = lb['Score'].min(), lb['Score'].max()
    lb['NormScore'] = (lb['Score'] - mn) / (mx - mn + 1e-8) + 0.1
    return dict(zip(lb['TeamName'], lb['NormScore']))

score_map = load_score_map(LEADERBOARD_CSV)
if score_map:
    print(f'リーダーボード: {len(score_map)} チーム読み込み済み（重み付きサンプリング有効）')
else:
    print('リーダーボードなし → 全勝者を均等学習')

def read_header(path):
    with open(path, 'rb') as f:
        chunk = f.read(2048).decode('utf-8', errors='ignore')
    agents  = re.findall(r'"Name":\s*"([^"]+)"', chunk)
    m       = re.search(r'"rewards":\s*\[([^\]]+)\]', chunk)
    rewards = []
    if m:
        for x in m.group(1).split(','):
            try: rewards.append(int(x.strip()))
            except: rewards.append(0)
    return agents, rewards

class PTCGDataset(Dataset):
    def __init__(self, max_files=None):
        raw_states, raw_masks, raw_targets, raw_weights = [], [], [], []
        json_files = sorted(glob.glob(f'{JSON_DIRECTORY}/*.json'))
        if max_files:
            json_files = json_files[:max_files]
        print(f'解析対象: {len(json_files)} ファイル')

        skipped_unmatched = skipped_no_state = skipped_invalid = matched = 0

        for path in tqdm(json_files, desc='JSONパース中'):
            try:
                agents, rewards = read_header(path)
            except Exception:
                continue

            if len(rewards) < 2: continue
            if   rewards[0] == 1: wi, wname = 0, (agents[0].strip() if agents else '')
            elif rewards[1] == 1: wi, wname = 1, (agents[1].strip() if len(agents)>1 else '')
            else: continue

            if score_map is not None:
                weight = score_map.get(wname, 0.0)
                if weight == 0.0:
                    skipped_unmatched += 1
                    continue
            else:
                weight = 1.0

            matched += 1
            try:
                with open(path, 'r') as f:
                    log = json.load(f)
            except Exception:
                continue

            for step in log.get('steps', []):
                if len(step) <= wi: continue
                ps = step[wi]
                sv = extract_state_vector(ps)
                if sv is None: skipped_no_state += 1; continue

                action = ps.get('action')
                if not action: continue
                obs = ps.get('observation', {})
                sel = obs.get('select')
                if not sel or 'option' not in sel: continue

                valid = len(sel['option'])
                target = action[0]
                if target >= MAX_ACTIONS or target >= valid:
                    skipped_invalid += 1; continue

                raw_states.append(sv)
                raw_masks.append(np.arange(MAX_ACTIONS) >= valid)
                raw_targets.append(target)
                raw_weights.append(weight)

        print(f'マッチ試合数     : {matched}')
        print(f'スキップ(未登録) : {skipped_unmatched}')
        print(f'有効サンプル数   : {len(raw_states)}')

        # メモリ効率化: スタック後に元リストを解放
        all_states = np.stack(raw_states);  del raw_states
        self.mean  = all_states.mean(0)
        self.std   = all_states.std(0)
        self.std[self.std == 0] = 1e-8
        np.savez(NORM_PATH, mean=self.mean, std=self.std)
        print(f'正規化パラメータ保存: {NORM_PATH}')

        norm = (all_states - self.mean) / self.std;  del all_states
        self.states  = torch.tensor(norm,                  dtype=torch.float32);  del norm
        self.masks   = torch.tensor(np.stack(raw_masks),   dtype=torch.bool)
        self.targets = torch.tensor(raw_targets,           dtype=torch.long)
        self.weights = torch.tensor(raw_weights,           dtype=torch.float32)

    def __len__(self):       return len(self.states)
    def __getitem__(self, i): return self.states[i], self.masks[i], self.targets[i]

dataset = PTCGDataset(max_files=MAX_FILES)

In [ ]:
# ============================================================
# Step 6: DataLoader & モデル初期化
# ============================================================
import torch.optim as optim

val_size   = int(len(dataset) * 0.2)
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_weights = dataset.weights[train_ds.indices]
sampler = WeightedRandomSampler(train_weights, len(train_weights), replacement=True)

nw = 2 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=nw, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=nw, pin_memory=torch.cuda.is_available())

print(f'学習: {train_size} / 検証: {val_size} サンプル')

model     = PTCGNet(INPUT_DIM, dropout=DROPOUT_RATE).to(device)
criterion = nn.CrossEntropyLoss()   # label_smoothing 削除（masked_fill(-1e9) と組み合わせると loss が異常値になる）
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

total_params = sum(p.numel() for p in model.parameters())
print(f'モデルパラメータ数: {total_params:,}')

In [ ]:
# ============================================================
# Step 7: 学習ループ
# ============================================================
print('== 学習開始 ==')
best_val_loss = float('inf')
best_val_acc  = 0.0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}

for epoch in range(EPOCHS):
    # --- 学習 ---
    model.train()
    tl = tc = tt = 0
    for states, masks, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]', leave=False):
        states, masks, targets = states.to(device), masks.to(device), targets.to(device)
        out  = model(states).masked_fill(masks, -1e9)
        loss = criterion(out, targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item() * states.size(0)
        tc += (out.argmax(1) == targets).sum().item()
        tt += states.size(0)

    # --- 検証 ---
    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for states, masks, targets in val_loader:
            states, masks, targets = states.to(device), masks.to(device), targets.to(device)
            out  = model(states).masked_fill(masks, -1e9)
            loss = criterion(out, targets)
            vl += loss.item() * states.size(0)
            vc += (out.argmax(1) == targets).sum().item()
            vt += states.size(0)

    t_loss, v_loss = tl/tt, vl/vt
    t_acc,  v_acc  = tc/tt*100, vc/vt*100
    cur_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    history['lr'].append(cur_lr)

    print(f'Epoch {epoch+1:2d}/{EPOCHS}  '
          f'train loss={t_loss:.4f} acc={t_acc:.1f}%  |  '
          f'val loss={v_loss:.4f} acc={v_acc:.1f}%  |  lr={cur_lr:.2e}')

    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_val_acc  = v_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  → ベストモデル保存 (val_loss={v_loss:.4f}, val_acc={v_acc:.1f}%)')

best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
print(f'\n== 学習完了 ==')
print(f'ベストエポック: {best_ep}  val_acc={best_val_acc:.1f}%')

In [ ]:
# ============================================================
# Step 8: 学習曲線の可視化
# ============================================================
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('PTCG Weighted Imitation Learning', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', ms=5)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val',   ms=5)
axes[0].axvline(best_ep, color='g', ls='--', label=f'Best={best_ep}')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss Curve'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train', ms=5)
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val',   ms=5)
axes[1].axvline(best_ep, color='g', ls='--', label=f'Best={best_ep}')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy Curve'); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(epochs_range, history['lr'], 'g-o', ms=5)
axes[2].set(xlabel='Epoch', ylabel='Learning Rate', title='LR Schedule', yscale='log'); axes[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'グラフ保存: {PLOT_PATH}')

In [ ]:
# ============================================================
# Step 9: 出力ファイルの確認
# ============================================================
print('== /kaggle/working/ の出力ファイル ==')
for f in os.listdir(WORKING):
    size = os.path.getsize(f'{WORKING}/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print()
print('学習後にこれらのファイルをダウンロードして')
print('submission/ フォルダに配置してください:')
print('  ptcg_baseline_model.pth  → submission/ptcg_baseline_model.pth')
print('  ptcg_normalization.npz   → submission/ptcg_normalization.npz')